# Quant Alpha Research: Comparing XGBoost, Transformers, HMM Regimes, and Reinforcement Learning

## Project Overview

This project compares classical machine-learning methods, transformer-based sequence models, hidden Markov model regime detection, and reinforcement-learning trading policies for detecting and trading short-term market alpha.

The project does not assume that transformers, HMMs, or reinforcement learning will automatically outperform simpler models. The purpose is to determine whether each additional layer of complexity produces measurable out-of-sample improvement over a strong XGBoost baseline.

The first version focuses on predicting meaningful intraday NVDA price movements using synchronized five-minute market data.

---

# Research Objective

The primary research question is:

**Can transformer models, HMM-based regime detection, and reinforcement-learning-based trading policies improve predictive and trading performance compared with a strong XGBoost baseline?**

The systems will be evaluated using:

* Classification performance
* Per-class precision, recall, and F1 score
* Macro F1 score
* Probability calibration
* Log loss
* Brier score
* Trading PnL
* Transaction-cost-adjusted returns
* Sharpe ratio
* Maximum drawdown
* Turnover
* Out-of-sample stability

The project will separately test whether improvements come from:

* Transformer sequence modeling
* Volume and activity features
* Context assets
* Time-of-day and weekday features
* HMM regime features
* Reinforcement-learning trade management

---

# Assets

## Primary Trading Target

* NVDA

## Context Assets

* QQQ
* SPY

NVDA is the asset being predicted and traded.

QQQ and SPY are included as context assets to help distinguish:

* NVDA-specific movement
* Technology-market movement
* Broad-market movement

Additional assets may be tested later as separate ablation experiments, but the first version uses only NVDA, QQQ, and SPY.

---

# Data Source and Trading Timeline

The first version uses Alpaca SIP five-minute candles.

## Base Configuration

* Data source: Alpaca SIP
* Candle size: 5 minutes
* Primary target: NVDA
* Context assets: QQQ and SPY
* Trading style: intraday
* Prediction horizon: 12 candles
* Prediction window: 60 minutes
* Overnight positions: disabled
* Regular market open: 6:30 AM Pacific Time
* Regular market close: 1:00 PM Pacific Time
* Final regular-session candle start: 12:55 PM Pacific Time
* Daily trading-state reset: enabled

Each complete trading day contains:

* 21 pre-open context candles
* 78 regular-session candles
* 99 total aligned candles per asset

The pre-open context period begins approximately 105 minutes before the regular market open.

---

# Pre-Open Context Candles

The dataset retains 21 five-minute candles before the regular market open.

These candles are no longer treated as EMA warmup candles because EMA features are excluded from the first model version.

Instead, the pre-open candles are used as historical market context for:

* Transformer input sequences
* Previous-candle return calculations
* Previous-candle volume changes
* Price-action history
* Cross-asset history

The model is not allowed to trade during the pre-open context period.

Pre-open rows:

* May appear inside transformer input sequences
* May be used to calculate causal historical features
* Do not receive supervised prediction labels
* Do not receive trading rewards
* Do not create trades

The existing code may still label these rows with:

```text
label_status = warmup
```

Conceptually, these rows represent:

```text
pre-open context
```

rather than EMA warmup.

---

# Prediction Target

The first version frames the problem as a three-class triple-barrier classification task.

At the close of each valid regular-session candle `t`, the model predicts whether NVDA will hit a positive or negative one-percent barrier during the next 12 candles.

The model outputs:

* `P(neutral)`
* `P(up)`
* `P(down)`

## Execution and Reference Price

The model observes all information through the close of candle `t`.

Because the signal is generated after candle `t` closes, the earliest realistic execution price is:

```text
NVDA open at candle t+1
```

The triple-barrier reference price is therefore the next candle’s open.

For each prediction candle `t`:

```text
entry_price = NVDA open at t+1
upper_barrier = entry_price × 1.01
lower_barrier = entry_price × 0.99
```

The future evaluation window consists of:

```text
candles t+1 through t+12
```

## Label Definitions

### Up

The upper one-percent barrier is reached before the lower barrier.

### Down

The lower one-percent barrier is reached before the upper barrier.

### Neutral

Neither barrier is reached during the following 12 candles.

### Ambiguous Same-Bar Event

If the upper and lower barriers are both reached for the first time inside the same five-minute candle, the ordering cannot be determined from OHLC data.

These rows receive:

```text
label_status = ambiguous_same_bar
```

They do not receive a target class.

## Class-ID Mapping

The current classification mapping is:

```text
neutral = 0
up      = 1
down    = 2
```

These values are category identifiers only. They do not imply a numerical relationship between the classes.

---

# Handling Ambiguous Labels

Ambiguous rows are not included as supervised prediction targets because their correct directional class is unknown.

No classification loss is calculated for an ambiguous target row.

However, the underlying OHLCV market data remains valid. Therefore, an ambiguous candle may still appear inside the historical input sequence for a later valid prediction target.

For example:

```text
Candle 30 target: ambiguous
Candle 35 target: valid up
```

The sequence ending at candle 35 may include candle 30’s market features.

That is valid because:

* Candle 30’s OHLCV data is known
* Candle 30’s target label is not used as an input feature
* Loss is calculated only against candle 35’s valid target

Supervised model training uses only rows where:

```text
label_status = valid
```

---

# Current Labeled Dataset Status

The current aligned and labeled dataset contains:

* Complete trading days: 1,539
* Total rows: 152,361
* Total labelable regular-session rows: 101,574
* Valid supervised targets: 101,535
* Ambiguous same-bar targets: 39
* Pre-open context rows: 32,319
* Incomplete future-window rows: 18,468

## Current Class Distribution

Among valid targets:

* Neutral: 64,964 rows, approximately 63.98%
* Down: 19,100 rows, approximately 18.81%
* Up: 17,471 rows, approximately 17.21%

The ambiguous rate is extremely low, so ambiguous target rows can be excluded without materially reducing the dataset.

Entire days containing ambiguous rows will not be removed because the market data on those days remains valid.

---

# Data Cleaning and Session Validation

Before labeling, feature creation, training, or backtesting, the pipeline validates each trading day.

A day is retained only when every required asset has all expected aligned candles.

## Required Assets in Version One

* NVDA
* QQQ
* SPY

## Complete-Day Requirements

Each asset must have:

* 21 pre-open context candles
* 78 regular-session candles
* 99 total candles
* No duplicate timestamps
* No missing required values
* Identical aligned timestamps across assets

Days that fail these checks are removed before alignment.

The aligned dataset is created using exact timestamp matching across NVDA, QQQ, and SPY.

---

# Labeling Restrictions

The final 12 regular-session candles of each day cannot receive new supervised labels because their full future 60-minute windows would extend beyond the regular trading session.

These rows receive:

```text
label_status = incomplete_future_window
```

The first version does not permit:

* Overnight labels
* Overnight trades
* Prediction windows crossing into the next trading day
* Labels based on incomplete future windows

---

# Feature Design Principles

The first feature set focuses on normalized, causal, and relatively stationary information.

Raw price levels will not be used directly as model inputs because NVDA’s absolute price changes substantially across the historical period.

All features must be calculable using information available at or before the close of the current candle.

Future information may be used only to construct labels, never to construct model inputs.

If any scalers, averages, medians, or normalization statistics are used, they must be fitted using training data only.

Validation and test data must use the values fitted on the training period.

---

# No EMA Features in Version One

EMA features are excluded from the first version.

The available dataset begins only 21 candles before the regular market open. Although an EMA could technically be calculated, the earliest pre-open EMA values would be poorly initialized because earlier historical candles are unavailable.

Feeding those inconsistent EMA values into transformer sequences could introduce undesirable artifacts.

The first version will therefore not include:

* EMA 9
* EMA 20
* EMA crossovers
* EMA slopes
* EMA distances
* EMA spread
* Bars since EMA crossover

This decision allows all transformer input candles to use a consistent and reliable feature set.

EMA features may be reconsidered only as a later optional experiment if a reliable initialization method becomes available.

---

# Core Per-Candle Feature Set

Both XGBoost and the transformer will use the same underlying per-candle market information.

## Candle and Price-Action Features

For NVDA, QQQ, and SPY, useful normalized candle features include:

```text
log_return_1
body_pct
range_pct
upper_wick_pct
lower_wick_pct
close_position_in_range
vwap_distance
```

Example definitions:

```text
log_return_1 = log(close_t / close_t-1)

body_pct = (close - open) / open

range_pct = (high - low) / open

upper_wick_pct =
    (high - max(open, close)) / open

lower_wick_pct =
    (min(open, close) - low) / open

close_position_in_range =
    (close - low) / (high - low)

vwap_distance =
    (close - vwap) / close
```

A safe fallback must be used when the candle range is zero.

---

# Volume and Market-Activity Features

The first version will be volume-aware rather than purely volume-based.

Volume alone may indicate unusual activity but usually does not determine price direction. It will therefore be combined with price-action and cross-asset information.

Core volume and activity features include:

```text
log_volume
log_trade_count
average_trade_size
dollar_volume
log_dollar_volume
volume_change_1
trade_count_change_1
```

Example definitions:

```text
log_volume = log(1 + volume)

log_trade_count = log(1 + trade_count)

average_trade_size =
    volume / max(trade_count, 1)

dollar_volume =
    volume × vwap

log_dollar_volume =
    log(1 + dollar_volume)

volume_change_1 =
    log((volume_t + 1) / (volume_t-1 + 1))
```

Log transformations help reduce the extreme skew found in raw volume and trade-count data.

---

# Time-Adjusted Relative Volume

Volume follows a strong intraday pattern.

Volume is normally high near the market open, lower around midday, and often higher near the close.

Because of this, raw volume should not be interpreted identically at every time of day.

A useful feature is:

```text
time_adjusted_relative_volume =
    current volume
    divided by
    typical training-period volume for the same asset and time of day
```

For example:

```text
NVDA volume at 6:35 AM
÷
median NVDA 6:35 AM volume from the training period
```

The time-of-day volume baseline must be calculated using training data only.

Validation and test data must use the training-fitted baseline.

This prevents future information from leaking into the model.

---

# Cross-Asset Context Features

QQQ and SPY help determine whether an NVDA move is stock-specific or part of a broader market move.

Useful context features include:

```text
NVDA log return
QQQ log return
SPY log return

NVDA return minus QQQ return
NVDA return minus SPY return

QQQ return minus SPY return
```

The model may also compare time-adjusted activity levels across assets after each asset has been independently normalized.

Raw volume differences between NVDA, QQQ, and SPY should generally not be used because each asset naturally trades at a different scale.

---

# Time-of-Day Features

The behavior of the market changes throughout the trading day.

The model will receive a normalized time-of-day feature based on the number of minutes since the regular market open.

```text
minutes_since_open =
    candle time in minutes
    minus
    6:30 AM in minutes
```

Examples:

```text
4:45 AM  → -105 minutes
6:30 AM  → 0 minutes
8:10 AM  → 100 minutes
12:55 PM → 385 minutes
```

The normalized feature is:

```text
time_of_day =
    minutes_since_open / 390
```

Premarket values are negative.

Regular-session values range from approximately zero to one.

A linear time-of-day feature is preferred initially because the start and end of the regular session should not necessarily be treated as adjacent.

The raw absolute timestamp and raw calendar date will not be fed into the models.

---

# Day-of-Week Features

The model will also receive the weekday.

Weekday values will be one-hot encoded:

```text
weekday_monday
weekday_tuesday
weekday_wednesday
weekday_thursday
weekday_friday
```

The raw weekday integer will not be used directly because it could incorrectly suggest that Friday is numerically greater than Monday.

All candles from the same trading day will share the same weekday values.

---

# Historical Information and Model Fairness

The transformer and XGBoost will use the same underlying market information but represent history differently.

## Transformer Representation

The transformer receives an ordered fixed-length sequence of per-candle feature rows.

Initial candidate sequence length:

```text
20 candles
```

Each prediction sequence ends at the current candle `t`.

For a target at candle `t`:

```text
Input:
the previous 19 candles plus candle t

Target:
the triple-barrier label attached to candle t
```

Because the dataset includes 21 pre-open context candles, the transformer can create a full sequence for predictions beginning at the regular market open.

A prediction based on the 6:30 AM candle is generated after that candle closes and may execute at the 6:35 AM open.

## XGBoost Representation

XGBoost receives tabular features for the current prediction point.

To create a strong baseline, XGBoost may also receive causal lagged summaries derived from the same historical window available to the transformer.

Possible XGBoost lagged features include:

```text
return lag 1
return lag 3
return lag 6
return lag 12
cumulative return over recent candles
volume change lags
recent maximum range
recent average log volume
```

This gives XGBoost access to comparable historical information without giving it future data.

---

# Transformer Target Selection and Loss

Not every row becomes a transformer training target.

A transformer sequence is included as a supervised training example only when:

* The target row is during the regular session
* The target row has `label_status = valid`
* A complete input sequence exists
* Every feature in the sequence is available
* The sequence does not cross into a different trading day

Loss is calculated only against valid target labels.

Examples:

```text
Sequence ending on valid up target
→ included, calculate loss against up

Sequence ending on ambiguous target
→ excluded, no loss calculated

Sequence ending on incomplete future-window target
→ excluded, no loss calculated

Sequence ending on pre-open target
→ excluded, no loss calculated
```

An ambiguous or non-target candle may still appear inside the historical sequence for a later valid target.

---

# Class Imbalance

The current target distribution contains a majority neutral class.

Accuracy alone is not sufficient because a model could obtain high accuracy by frequently predicting neutral.

The models will be evaluated using:

* Confusion matrix
* Per-class precision
* Per-class recall
* Per-class F1 score
* Macro F1 score
* Log loss
* Brier score
* Probability calibration
* Performance on up and down events specifically

If necessary, the project may test:

* Class-weighted loss
* Balanced sampling
* Validation-selected probability thresholds
* Focal loss for the transformer

All class-balancing decisions must be selected using training and validation data only.

---

# Probability Calibration

Because the models output probabilities, the reliability of those probabilities is important.

A model predicting:

```text
P(up) = 0.70
```

should ideally be correct close to 70 percent of the time across similar predictions.

Calibration will be evaluated using:

* Brier score
* Log loss
* Reliability curves
* Confidence buckets
* Expected calibration error

Calibrated probabilities are important for:

* Trade filtering
* Position sizing
* Rule-based thresholds
* Reinforcement-learning state inputs

---

# Methods

## 1. XGBoost Baseline

XGBoost is the primary classical machine-learning baseline.

It will use:

* Normalized candle features
* Volume and activity features
* Time-adjusted relative volume
* QQQ and SPY context features
* Time-of-day features
* Day-of-week features
* Causal lagged and summary features
* Optional HMM regime features

The XGBoost classifier outputs:

* `P(neutral)`
* `P(up)`
* `P(down)`

The XGBoost model establishes the performance level that more complex methods must beat.

---

## 2. Transformer-Based Predictor

The transformer learns from ordered sequences of recent five-minute candles.

Each sequence contains the same core per-candle market features used by the XGBoost pipeline.

The transformer does not receive EMA features in version one.

The purpose of the transformer is to test whether it can learn useful temporal relationships from:

* Recent price action
* Volume evolution
* Trade-count activity
* Relative movement across assets
* Intraday timing
* Weekday effects

The transformer outputs:

* `P(neutral)`
* `P(up)`
* `P(down)`

---

## 3. Hidden Markov Model Regime Detection

An HMM will be tested as an optional regime-feature generator.

The HMM will not replace XGBoost or the transformer.

It will attempt to identify latent market conditions such as:

* Trending
* Choppy
* High activity
* Low activity
* Risk-on
* Risk-off

Possible HMM inputs include:

* NVDA return
* QQQ return
* SPY return
* NVDA relative return versus QQQ
* NVDA relative return versus SPY
* Candle range percentage
* Log volume
* Time-adjusted relative volume
* Trade-count activity

Possible HMM outputs include:

```text
hmm_regime
hmm_regime_prob_0
hmm_regime_prob_1
hmm_regime_prob_2
regime_changed
bars_since_regime_change
```

The HMM must be trained only on the training period.

Validation and test regimes must be inferred using the training-fitted HMM without refitting on future data.

---

## 4. Signal-State Features

After a prediction model outputs probabilities, the trading layer may create signal-state features such as:

```text
p_up
p_down
p_neutral
p_edge
p_abs_edge
signal_direction
signal_age_bars
signal_active
p_edge_change_1
p_edge_rolling_mean
signal_persistence
```

Example:

```text
p_edge = p_up - p_down
```

These features allow the trading system to distinguish between:

* Strong and weak predictions
* Fresh and stale signals
* Persistent and temporary signals
* Increasing and decreasing confidence

---

## 5. Rule-Based Trading Layer

Before reinforcement learning, the project will evaluate simple probability-based trading rules.

Example rules include:

* Go long when `P(up)` exceeds a threshold and sufficiently exceeds `P(down)`
* Go short when `P(down)` exceeds a threshold and sufficiently exceeds `P(up)`
* Stay flat when confidence is weak
* Stay flat when probabilities are too close
* Avoid new entries too close to market close

Thresholds will be tuned using validation data only.

The final test set will not be used for rule selection or threshold tuning.

---

## 6. Reinforcement-Learning Trading Agent

The reinforcement-learning agent acts as the trading decision layer.

It does not replace the prediction models.

The RL agent receives:

* Market features
* Prediction probabilities
* Signal-state features
* Optional HMM regime features
* Portfolio-state features

## RL Market-State Features

Possible market-state inputs include:

* NVDA price-action features
* NVDA volume and activity features
* QQQ context features
* SPY context features
* Time-adjusted relative volume
* Time of day
* Day of week

## RL Model-Signal Features

* `p_neutral`
* `p_up`
* `p_down`
* `p_edge`
* Signal direction
* Signal persistence
* Signal age
* Probability momentum

## RL Portfolio-State Features

* Current position
* Entry price
* Bars in position
* Unrealized PnL
* Realized PnL
* Maximum favorable excursion
* Maximum adverse excursion
* Remaining bars until market close

The purpose of RL is to learn:

* Whether to act on a signal
* When to enter
* When to exit
* When to remain flat
* How to manage risk
* How to handle persistent or changing signals

---

# RL Episode Structure

Each RL episode represents one complete trading day.

At the start of each day:

* Position resets to flat
* Entry price resets
* Unrealized PnL resets
* Realized daily PnL resets
* Bars in position reset
* Signal-state counters reset

The agent may observe pre-open context candles but cannot trade during them.

Trading begins during the regular session.

At the end of the regular trading day:

* Any remaining position is closed
* No position is carried overnight
* The episode terminates

---

# RL Action Space

The initial RL environment will use a discrete target-position action space:

```text
0 = short
1 = flat
2 = long
```

Later experiments may test continuous position sizing.

---

# RL Reward Function

The first RL reward function will prioritize transaction-cost-adjusted changes in portfolio value.

Possible reward components include:

* Realized PnL
* Unrealized PnL change
* Slippage
* Fees
* Spread costs
* Turnover penalty
* Drawdown penalty
* Penalty for holding positions near market close

Complex reward shaping will be added only after the basic RL environment is working correctly.

---

# Execution Timing

To avoid lookahead bias:

* Candle `t` completes
* Features through candle `t` become available
* Prediction is generated after candle `t` closes
* Trading decision is generated after candle `t` closes
* Earliest execution occurs at the open of candle `t+1`

The model may not use candle `t+1` as an input feature when predicting candle `t`.

The next candle’s open may be used during label construction because labels describe future outcomes.

This distinction is essential:

```text
Future data in the label:
allowed

Future data in model features:
not allowed
```

---

# Costs and Slippage

Backtests will include realistic estimated trading costs.

The first version will model:

* Slippage
* Fees or commissions
* Optional spread costs
* Position-size constraints
* Turnover

Example conservative execution assumptions:

```text
Long entry:
next open adjusted upward by slippage

Long exit:
execution price adjusted downward by slippage

Short entry:
next open adjusted downward by slippage

Short exit:
execution price adjusted upward by slippage
```

Primary reported performance will be transaction-cost adjusted.

---

# Validation and Test Methodology

The dataset will be split chronologically.

The final test period must remain locked until the methodology is finalized.

Because the available dataset may not contain exactly ten full years, the split will be based on the actual available date range rather than assuming a fixed number of years.

## Recommended Structure

* First portion of data: model development and walk-forward validation
* Final approximately 20 percent of dates: locked out-of-sample test period

Within the development period, expanding or rolling walk-forward folds will be used.

For every fold:

* Feature statistics are fitted on training data only
* Time-of-day volume baselines are fitted on training data only
* Scalers are fitted on training data only
* HMM is fitted on training data only
* XGBoost and transformer models are trained on training data only
* Validation data is used for tuning and model selection

## Purging and Embargo

Because neighboring target rows use overlapping future 12-candle windows, rows near train-validation or validation-test boundaries may share future information.

A purge or embargo of at least 12 candles should be applied around chronological split boundaries.

This helps prevent information from overlapping across dataset partitions.

---

# Model Selection and Hyperparameter Tuning

The final test set will not be used for:

* Feature selection
* Hyperparameter tuning
* Sequence-length selection
* Probability-threshold tuning
* Class-weight selection
* HMM regime-count selection
* RL reward tuning
* RL policy selection
* Backtest rule changes

Only after the methodology is locked will the final test set be evaluated.

---

# Non-ML Baselines

The project will compare the machine-learning systems against simple non-ML strategies.

Possible baselines include:

* Always flat
* Buy and hold NVDA during the test period
* Simple short-term momentum
* Simple short-term mean reversion
* Volume-breakout strategy
* Relative-volume strategy
* Random trading policy

These baselines help determine whether machine learning adds value beyond simple behavior.

---

# Ablation Studies

Ablation studies will determine which components create real improvement.

Important comparisons include:

* Without context assets versus with QQQ and SPY
* Without volume features versus with volume features
* Without time-adjusted relative volume versus with it
* Without time-of-day features versus with them
* Without weekday features versus with them
* Without HMM regime features versus with them
* XGBoost versus transformer
* Rule-based trading versus RL trading
* Different transformer sequence lengths
* Current-row XGBoost versus lag-enhanced XGBoost

EMA features are not part of the first-version ablation plan.

---

# Risk-Management Constraints

The backtester and RL environment will include explicit risk controls.

Initial controls may include:

* Maximum position size
* No overnight positions
* Forced flat before market close
* Maximum holding period
* Slippage and fee modeling
* Optional stop-loss
* Optional take-profit
* Optional maximum daily loss
* No new positions when insufficient trading time remains

---

# Main Model Comparison

## Without HMM Regime Features

1. XGBoost probabilities plus rule-based trading
2. Transformer probabilities plus rule-based trading
3. XGBoost probabilities plus RL trading agent
4. Transformer probabilities plus RL trading agent

## With HMM Regime Features

5. XGBoost plus HMM features plus rule-based trading
6. Transformer plus HMM features plus rule-based trading
7. XGBoost plus HMM features plus RL trading agent
8. Transformer plus HMM features plus RL trading agent

This structure answers:

1. Does the transformer improve prediction compared with XGBoost?
2. Does RL improve trading decisions compared with simple rules?
3. Do HMM regime features improve prediction or trading performance?

---

# Backtesting Assumptions

The first backtester will use the following explicit assumptions:

* Five-minute candles
* Intraday trading only
* No overnight positions
* Daily trading-state reset
* Pre-open candles used only as context
* No pre-open trading
* No pre-open supervised targets
* No new target when the complete future window is unavailable
* Signal generated after candle `t`
* Earliest execution at candle `t+1` open
* Slippage included
* Fees included
* Optional spread cost included
* Position limits included
* Forced flat before market close
* Chronological train, validation, and test splits
* Training-only fitting for all learned preprocessing statistics
* Ambiguous rows excluded as targets but retained as historical market context

---

# Reproducibility

Every experiment should record:

* Dataset version
* Included assets
* Feature configuration
* Sequence length
* Model architecture
* Hyperparameters
* Random seed
* Train-validation-test dates
* Purge and embargo settings
* Class weights
* Probability thresholds
* Backtest settings
* Slippage assumptions
* Fee assumptions
* Performance metrics

Every reported result should be reproducible from a saved configuration.

---

# Updated Project Workflow

1. Download or load Alpaca SIP candle data
2. Convert timestamps to Pacific Time
3. Retain 21 pre-open and 78 regular-session candles
4. Validate complete days for NVDA, QQQ, and SPY
5. Keep only dates complete across every required asset
6. Align assets on exact timestamps
7. Build triple-barrier labels using next-candle open as the reference price
8. Mark ambiguous same-bar targets
9. Mark final 12 candles as incomplete future-window targets
10. Save the complete labeled dataset
11. Build normalized candle features
12. Build volume and activity features
13. Build time-adjusted relative-volume features using training-only baselines
14. Build cross-asset context features
15. Build time-of-day and weekday features
16. Create chronological train, validation, and locked test partitions
17. Apply purge or embargo around split boundaries
18. Build XGBoost tabular datasets
19. Build transformer sequences ending only on valid target rows
20. Train XGBoost baseline
21. Train transformer predictor
22. Evaluate classification and calibration
23. Train optional HMM regime detector using training data only
24. Compare models with and without HMM features
25. Generate prediction probabilities
26. Build signal-state features
27. Tune rule-based trading thresholds using validation only
28. Build and train RL trading agents
29. Run transaction-cost-adjusted backtests
30. Compare all model and trading variants
31. Evaluate the locked final test period once the methodology is finalized

---

# Updated Initial Development Plan

## Phase 1: Data Pipeline — Completed

* Downloaded per-symbol Alpaca SIP data
* Converted timestamps to Pacific Time
* Retained 21 pre-open context candles
* Retained 78 regular-session candles
* Removed incomplete days
* Intersected complete days across symbols
* Aligned NVDA, QQQ, and SPY by timestamp

## Phase 2: Triple-Barrier Labeling — Completed

* Used next-candle open as the barrier reference
* Evaluated the next 12 candles
* Assigned neutral, up, or down based on first barrier reached
* Marked same-candle double hits as ambiguous
* Excluded the final 12 regular-session candles from new labels
* Stored bars-to-barrier for up and down events
* Verified each day has 66 labelable targets

## Phase 3: Feature Engineering — Next

Build the first causal feature set:

* Log returns
* Candle body percentage
* Candle range percentage
* Upper and lower wick percentages
* Close position inside candle range
* VWAP distance
* Log volume
* Log trade count
* Average trade size
* Dollar volume
* Volume and trade-count changes
* Time-adjusted relative volume
* QQQ and SPY context features
* NVDA relative returns
* Normalized time of day
* One-hot weekday features

Do not include EMA features.

## Phase 4: Chronological Dataset Splitting

* Lock the final out-of-sample test period
* Create walk-forward development folds
* Add a 12-candle purge or embargo near split boundaries
* Fit all preprocessing statistics on training data only

## Phase 5: XGBoost Baseline

* Build a strong lag-enhanced tabular baseline
* Predict neutral, up, and down probabilities
* Evaluate class performance and calibration
* Tune using validation data only

## Phase 6: Transformer Predictor

* Build fixed-length sequences from per-candle features
* Use pre-open candles as sequence context
* Include only valid target rows in the supervised loss
* Exclude ambiguous rows as targets
* Compare multiple sequence lengths using validation data

## Phase 7: HMM Regime Detection

* Train HMM using training data only
* Generate regime probabilities and regime-state features
* Compare XGBoost and transformer performance with and without regimes

## Phase 8: Rule-Based Trading Baseline

* Build probability-threshold trading rules
* Include realistic costs and slippage
* Tune thresholds using validation data only

## Phase 9: Reinforcement Learning

* Build daily trading episodes
* Use market state, model probabilities, and portfolio state
* Prevent pre-open and overnight trading
* Train RL policies without accessing the final test period

## Phase 10: Final Evaluation

* Compare all major model and trading variants
* Run ablation studies
* Evaluate calibration and prediction quality
* Evaluate transaction-cost-adjusted trading performance
* Run the locked final test period once the methodology is fixed


### imports of major libraries and scripts

In [1]:
#autoreload 
%load_ext autoreload
%autoreload 2
# Core Python
import os
from datetime import datetime, timedelta
from pathlib import Path

# Environment variables
from dotenv import load_dotenv

# Data handling
import numpy as np
import pandas as pd
import polars as pl

# Plotting
import plotly.graph_objects as go
import plotly.express as px

# Progress bars
from tqdm.auto import tqdm

# Alpaca market data
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.enums import DataFeed
from alpaca.data.requests import StockLatestTradeRequest

# Project paths
from quant_research.utils.paths import (
    PROJECT_ROOT,
    DATA_DIR,
    RAW_DATA_DIR,
    ALPACA_RAW_DIR,
    PROCESSED_DATA_DIR,
    NOTEBOOKS_DIR,
    RESULTS_DIR,
    LOGS_DIR,
    ensure_project_dirs,
)

# Setup
load_dotenv(PROJECT_ROOT / ".env")
ensure_project_dirs()

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Project root:", PROJECT_ROOT)
print("Raw Alpaca data:", ALPACA_RAW_DIR)
print("Processed data:", PROCESSED_DATA_DIR)

/Users/rchbeir/Desktop/Quant work/quant_proj/proj/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/rchbeir/Desktop/Quant work/quant_proj
Raw Alpaca data: /Users/rchbeir/Desktop/Quant work/quant_proj/data/raw/alpaca
Processed data: /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed


### Now we are testing the Alpaca connection

In [2]:
from quant_research.data.data_downloader import CheckConnection
CheckConnection()

Alpaca connected


### Downloading the actual data

In [3]:
from datetime import datetime
from zoneinfo import ZoneInfo

from quant_research.utils.paths import ALPACA_RAW_DIR
from quant_research.data.data_downloader import Download_5_min_Data

symbols = ["NVDA", "QQQ", "SPY"]
# Alpaca expects timezone-aware datetimes.
# This requests roughly 10 years of historical 5-minute data.
# Format: datetime(year, month, day, hour, minute, tzinfo=ZoneInfo("UTC"))
# 14:30 UTC is 6:30 AM Pacific during standard time, which is the U.S. market open.
# The end date is exclusive, so this covers from Jan 2, 2016 up to Jan 2, 2026.

start = datetime(2016, 1, 2, 14, 30, tzinfo=ZoneInfo("UTC"))
end = datetime(2026, 1, 2, 21, 0, tzinfo=ZoneInfo("UTC"))

df = Download_5_min_Data(
    symbols=symbols,
    start=start,
    end=end,
    timeframe="5min",
    directory=ALPACA_RAW_DIR,
)


NVDA: file already exists -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/raw/alpaca/NVDA_5min.parquet
Would you like to overwrite it? (y/n):  


Please answer with 'y' or 'n'.


NVDA: file already exists -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/raw/alpaca/NVDA_5min.parquet
Would you like to overwrite it? (y/n):  n


NVDA: skipping.


QQQ: file already exists -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/raw/alpaca/QQQ_5min.parquet
Would you like to overwrite it? (y/n):  n


QQQ: skipping.


SPY: file already exists -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/raw/alpaca/SPY_5min.parquet
Would you like to overwrite it? (y/n):  n


SPY: skipping.


### Now we have to check for overlapping days and store them

In [9]:
from quant_research.data.data_processor import save_overlapping_days
save_overlapping_days(symbols)

Transformed NVDA to pacific time
outside candles removed
NVDA has 2515 total days after time filtering
NVDA has 1610 complete days
Transformed QQQ to pacific time
outside candles removed
QQQ has 2515 total days after time filtering
QQQ has 2203 complete days
Transformed SPY to pacific time
outside candles removed
SPY has 2515 total days after time filtering
SPY has 2462 complete days
initial cleaning complete, there are 1539 in common


NVDA: file does not exist -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/NVDA_5min.parquet
Would you like to save processed data it? (y/n):  y


NVDA: proceeding.
Saved processed NVDA: 152,361 rows -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/NVDA_5min.parquet


QQQ: file does not exist -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/QQQ_5min.parquet
Would you like to save processed data it? (y/n):  y


QQQ: proceeding.
Saved processed QQQ: 152,361 rows -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/QQQ_5min.parquet


SPY: file does not exist -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/SPY_5min.parquet
Would you like to save processed data it? (y/n):  y


SPY: proceeding.
Saved processed SPY: 152,361 rows -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/SPY_5min.parquet


In [12]:
#quick sanity check to see if they have all the same number of row
for symbol in symbols:
    path = PROCESSED_DATA_DIR / f"{symbol}_5min.parquet"
    df = pd.read_parquet(path)
    print(symbol)
    print("shape:", df.shape)
    print("unique days:", df["date"].nunique())
    print("start:", df["timestamp"].min())
    print("end:", df["timestamp"].max())
    print()
df.head()

NVDA
shape: (152361, 12)
unique days: 1539
start: 2016-02-18 12:45:00+00:00
end: 2026-01-02 20:55:00+00:00

QQQ
shape: (152361, 12)
unique days: 1539
start: 2016-02-18 12:45:00+00:00
end: 2026-01-02 20:55:00+00:00

SPY
shape: (152361, 12)
unique days: 1539
start: 2016-02-18 12:45:00+00:00
end: 2026-01-02 20:55:00+00:00



,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,timestamp_pt,date,time
0,SPY,2016-02-18 12:45:00+00:00,193.34,193.34,193.19,193.24,89213.0,113.0,193.242909,2016-02-18 04:45:00-08:00,2016-02-18,04:45:00
1,SPY,2016-02-18 12:50:00+00:00,193.24,193.36,193.22,193.35,72778.0,90.0,193.309160,2016-02-18 04:50:00-08:00,2016-02-18,04:50:00
2,SPY,2016-02-18 12:55:00+00:00,193.36,193.46,193.35,193.44,34285.0,67.0,193.426847,2016-02-18 04:55:00-08:00,2016-02-18,04:55:00
3,SPY,2016-02-18 13:00:00+00:00,193.41,193.65,193.22,193.40,86180.0,140.0,193.416101,2016-02-18 05:00:00-08:00,2016-02-18,05:00:00
4,SPY,2016-02-18 13:05:00+00:00,193.40,193.57,193.38,193.45,76086.0,166.0,193.486128,2016-02-18 05:05:00-08:00,2016-02-18,05:05:00


In [13]:
from quant_research.data.data_aligner import build_aligned_dataset

symbols = ["NVDA", "QQQ", "SPY"]

aligned = build_aligned_dataset(
    symbols=symbols,
    timeframe="5min",
)

aligned.head()

Loading and aligning NVDA...
Loading and aligning QQQ...
Loading and aligning SPY...
Saved aligned dataset: 152,361 rows -> /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/aligned_5min.parquet


,timestamp,NVDA_open,NVDA_high,NVDA_low,NVDA_close,NVDA_volume,NVDA_trade_count,NVDA_vwap,timestamp_pt,date,time,QQQ_open,QQQ_high,QQQ_low,QQQ_close,QQQ_volume,QQQ_trade_count,QQQ_vwap,SPY_open,SPY_high,SPY_low,SPY_close,SPY_volume,SPY_trade_count,SPY_vwap
0,2016-02-18 12:45:00+00:00,29.82,29.82,29.68,29.73,5504.0,17.0,29.755625,2016-02-18 04:45:00-08:00,2016-02-18,04:45:00,102.87,102.87,102.83,102.83,3212.0,7.0,102.859408,193.34,193.34,193.19,193.24,89213.0,113.0,193.242909
1,2016-02-18 12:50:00+00:00,29.73,29.74,29.62,29.74,4408.0,28.0,29.720000,2016-02-18 04:50:00-08:00,2016-02-18,04:50:00,102.83,102.90,102.83,102.90,3950.0,8.0,102.870633,193.24,193.36,193.22,193.35,72778.0,90.0,193.309160
2,2016-02-18 12:55:00+00:00,29.74,29.74,29.74,29.74,1035.0,3.0,29.740000,2016-02-18 04:55:00-08:00,2016-02-18,04:55:00,102.91,102.91,102.91,102.91,1400.0,3.0,102.910000,193.36,193.46,193.35,193.44,34285.0,67.0,193.426847
3,2016-02-18 13:00:00+00:00,29.70,30.03,29.66,29.88,39864.0,91.0,29.792445,2016-02-18 05:00:00-08:00,2016-02-18,05:00:00,102.95,102.95,102.90,102.95,38699.0,42.0,102.932448,193.41,193.65,193.22,193.40,86180.0,140.0,193.416101
4,2016-02-18 13:05:00+00:00,29.86,29.90,29.80,29.80,5635.0,26.0,29.844374,2016-02-18 05:05:00-08:00,2016-02-18,05:05:00,102.95,103.01,102.94,103.00,15499.0,40.0,102.971399,193.40,193.57,193.38,193.45,76086.0,166.0,193.486128


In [5]:
import importlib
import quant_research.data.label_builder as data_labeler


labeled = data_labeler.label_data(
    timeframe="5min",
    pct=0.01,
    Horizon_bars=12,
)

labeled.head()

target_name
neutral    64964
<NA>       50826
down       19100
up         17471
Name: count, dtype: int64[pyarrow]
label_status
valid                       101535
warmup                       32319
incomplete_future_window     18468
ambiguous_same_bar              39
Name: count, dtype: int64
target_name
neutral    0.639819
down       0.188112
up         0.172069
Name: proportion, dtype: double[pyarrow]
66    1539
Name: count, dtype: int64
Ambiguous rate: 0.0384%
Average bars to hit barrier: 6.07
Average minutes to hit barrier: 30.35

Average bars to barrier by direction:
target_name
down    5.967801
up      6.181043
Name: bars_to_barrier, dtype: Float64

Average minutes to barrier by direction:
target_name
down    29.839005
up      30.905214
Name: bars_to_barrier, dtype: Float64
Saved labeled dataset to /Users/rchbeir/Desktop/Quant work/quant_proj/data/processed/labeled_5min.parquet


,timestamp,NVDA_open,NVDA_high,NVDA_low,NVDA_close,NVDA_volume,NVDA_trade_count,NVDA_vwap,timestamp_pt,date,time,QQQ_open,QQQ_high,QQQ_low,QQQ_close,QQQ_volume,QQQ_trade_count,QQQ_vwap,SPY_open,SPY_high,SPY_low,SPY_close,SPY_volume,SPY_trade_count,SPY_vwap,target_name,target_class,bars_to_barrier,label_status
0,2016-02-18 12:45:00+00:00,29.82,29.82,29.68,29.73,5504.0,17.0,29.755625,2016-02-18 04:45:00-08:00,2016-02-18,04:45:00,102.87,102.87,102.83,102.83,3212.0,7.0,102.859408,193.34,193.34,193.19,193.24,89213.0,113.0,193.242909,<NA>,<NA>,<NA>,warmup
1,2016-02-18 12:50:00+00:00,29.73,29.74,29.62,29.74,4408.0,28.0,29.720000,2016-02-18 04:50:00-08:00,2016-02-18,04:50:00,102.83,102.90,102.83,102.90,3950.0,8.0,102.870633,193.24,193.36,193.22,193.35,72778.0,90.0,193.309160,<NA>,<NA>,<NA>,warmup
2,2016-02-18 12:55:00+00:00,29.74,29.74,29.74,29.74,1035.0,3.0,29.740000,2016-02-18 04:55:00-08:00,2016-02-18,04:55:00,102.91,102.91,102.91,102.91,1400.0,3.0,102.910000,193.36,193.46,193.35,193.44,34285.0,67.0,193.426847,<NA>,<NA>,<NA>,warmup
3,2016-02-18 13:00:00+00:00,29.70,30.03,29.66,29.88,39864.0,91.0,29.792445,2016-02-18 05:00:00-08:00,2016-02-18,05:00:00,102.95,102.95,102.90,102.95,38699.0,42.0,102.932448,193.41,193.65,193.22,193.40,86180.0,140.0,193.416101,<NA>,<NA>,<NA>,warmup
4,2016-02-18 13:05:00+00:00,29.86,29.90,29.80,29.80,5635.0,26.0,29.844374,2016-02-18 05:05:00-08:00,2016-02-18,05:05:00,102.95,103.01,102.94,103.00,15499.0,40.0,102.971399,193.40,193.57,193.38,193.45,76086.0,166.0,193.486128,<NA>,<NA>,<NA>,warmup
